# College Admission Classification Labwork

Import Modules

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

Using pandas, load "College Admission Train Data.csv"

In [ ]:
# Le notebook fonctionne depuis la racine du dépôt comme depuis le dossier notebooks.
DATA_DIR = Path("data") if Path("data").is_dir() else Path("../data")

train_path = DATA_DIR / "College Admission Train Data.csv"
input_data = pd.read_csv(train_path)

expected_columns = {"Exam1", "Exam2", "Decision"}
if set(input_data.columns) != expected_columns:
    raise ValueError(f"Colonnes attendues : {sorted(expected_columns)}")

print(f"Jeu d'entraînement : {input_data.shape[0]} étudiants")
display(input_data.head())

---

In this section, we will ignore the previous jury decision.


Plot the dataset using scores only :

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(input_data["Exam1"], input_data["Exam2"], s=18, alpha=0.65)
plt.xlabel("Note à l'examen 1")
plt.ylabel("Note à l'examen 2")
plt.title("Notes des étudiants")
plt.xlim(0, 20)
plt.ylim(0, 20)
plt.grid(alpha=0.2)
plt.show()

Compute the decision for each student (admitted if mean > 10) and store it in an array named decision:

In [ ]:
decision = ((input_data["Exam1"] + input_data["Exam2"]) / 2 > 10).astype(int).to_numpy()

print("Décisions calculées (0 = refusé, 1 = admis) :")
print(pd.Series(decision).value_counts().sort_index())

Plot the dataset using decision as color :

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(
    input_data["Exam1"],
    input_data["Exam2"],
    c=decision,
    cmap="coolwarm",
    s=18,
    alpha=0.7,
)
plt.xlabel("Note à l'examen 1")
plt.ylabel("Note à l'examen 2")
plt.title("Décision calculée à partir de la moyenne")
plt.xlim(0, 20)
plt.ylim(0, 20)
plt.grid(alpha=0.2)
plt.show()

In [ ]:
def plot_decision_boundary(input_data, point_colors, x_min=0, x_max=20, y_min=0, y_max=20, plot_step=0.05):
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, plot_step),
        np.arange(y_min, y_max, plot_step),
    )
    zz = ((xx.ravel() + yy.ravel()) / 2 > 10).reshape(xx.shape)

    plt.figure(figsize=(7, 6))
    plt.pcolormesh(xx, yy, zz, cmap="Pastel1", shading="auto")
    plt.scatter(
        input_data["Exam1"],
        input_data["Exam2"],
        s=18,
        c=point_colors,
        cmap="coolwarm",
        edgecolors="none",
        alpha=0.75,
    )
    plt.axis([x_min, x_max, y_min, y_max])
    plt.xlabel("Note à l'examen 1")
    plt.ylabel("Note à l'examen 2")
    plt.show()

In [ ]:
plot_decision_boundary(input_data, decision)
plt.show()

Plot the dicision boundaries and the data points (Apply class color from the dataset decision) :

In [ ]:
# Le fond représente la règle "moyenne > 10" ; les points sont colorés selon le jury.
plot_decision_boundary(input_data, input_data["Decision"].to_numpy())

What is the algorithm decision accuracy related to the jury decision ??

In [ ]:
rule_accuracy = accuracy_score(input_data["Decision"], decision)
print(f"Précision de la règle moyenne > 10 par rapport au jury : {rule_accuracy:.2%}")

# Linear Discriminant Analysis

Create and fit the model

In [ ]:
features = ["Exam1", "Exam2"]
X_train = input_data[features]
y_train = input_data["Decision"]

lda_model = LinearDiscriminantAnalysis()
lda_model.fit(X_train, y_train)
lda_model

Display the data and the decision boundaries 

In [ ]:
def plot_model_decision_boundary(model, input_data, x_min=0, x_max=20, y_min=0, y_max=20, plot_step=0.1):
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, plot_step),
        np.arange(y_min, y_max, plot_step),
    )
    grid = pd.DataFrame(np.c_[xx.ravel(), yy.ravel()], columns=["Exam1", "Exam2"])
    zz = model.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(7, 6))
    plt.pcolormesh(xx, yy, zz, cmap="Pastel1", shading="auto")
    plt.scatter(
        input_data["Exam1"],
        input_data["Exam2"],
        s=18,
        c=input_data["Decision"],
        cmap="coolwarm",
        edgecolors="none",
        alpha=0.75,
    )
    plt.axis([x_min, x_max, y_min, y_max])
    plt.xlabel("Note à l'examen 1")
    plt.ylabel("Note à l'examen 2")
    plt.title(type(model).__name__)
    plt.show()

In [ ]:
plot_model_decision_boundary(lda_model, input_data)

Measure the accuracy of your model

In [ ]:
lda_train_accuracy = accuracy_score(y_train, lda_model.predict(X_train))
print(f"Précision LDA sur l'entraînement : {lda_train_accuracy:.2%}")

# Quadratic Discriminant Analysis

In [ ]:
qda_model = QuadraticDiscriminantAnalysis()
qda_model.fit(X_train, y_train)
qda_model

In [ ]:
plot_model_decision_boundary(qda_model, input_data)

In [ ]:
qda_train_accuracy = accuracy_score(y_train, qda_model.predict(X_train))
print(f"Précision QDA sur l'entraînement : {qda_train_accuracy:.2%}")

---
# Decisision Tree Classifier

In [ ]:
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)
tree_model

In [ ]:
plot_model_decision_boundary(tree_model, input_data)
tree_train_accuracy = accuracy_score(y_train, tree_model.predict(X_train))
print(f"Précision arbre de décision sur l'entraînement : {tree_train_accuracy:.2%}")

---

# Create a K-Nearest Neighbors Classifier (KNN)

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
knn_model

In [ ]:
plot_model_decision_boundary(knn_model, input_data)
knn_train_accuracy = accuracy_score(y_train, knn_model.predict(X_train))
print(f"Précision KNN sur l'entraînement : {knn_train_accuracy:.2%}")

# Random Forest

In [ ]:
forest_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
forest_model.fit(X_train, y_train)
forest_model

In [ ]:
plot_model_decision_boundary(forest_model, input_data)
forest_train_accuracy = accuracy_score(y_train, forest_model.predict(X_train))
print(f"Précision forêt aléatoire sur l'entraînement : {forest_train_accuracy:.2%}")

---

Load 'College Admission Test Data.csv'

In [ ]:
test_path = DATA_DIR / "College Admission Test Data.csv"
test_data = pd.read_csv(test_path)

if set(test_data.columns) != expected_columns:
    raise ValueError(f"Colonnes attendues : {sorted(expected_columns)}")

X_test = test_data[features]
y_test = test_data["Decision"]
print(f"Jeu de test : {test_data.shape[0]} étudiants")
display(test_data.head())

Compute and print the accuracy of all models on test data

In [ ]:
models = {
    "Règle moyenne > 10": None,
    "LDA": lda_model,
    "QDA": qda_model,
    "Arbre de décision": tree_model,
    "KNN (k=5)": knn_model,
    "Forêt aléatoire": forest_model,
}

test_rule = ((test_data["Exam1"] + test_data["Exam2"]) / 2 > 10).astype(int)
scores = {}
for name, model in models.items():
    predictions = test_rule if model is None else model.predict(X_test)
    scores[name] = accuracy_score(y_test, predictions)

results = (
    pd.Series(scores, name="Accuracy")
    .sort_values(ascending=False)
    .to_frame()
)
display(results.style.format({"Accuracy": "{:.2%}"}))
print(f"Meilleur modèle sur le jeu de test : {results.index[0]} ({results.iloc[0, 0]:.2%})")